# Notebook 04 — Residual Feedback Loop

**Repo:** `temporal-phase-lock`  
**Notebook:** `notebooks/04_residual_feedback_loop.ipynb`  
**Purpose:** detect weakest temporal-reasoning stage, revise it, and re-measure phase-lock.

Notebook 03 measured stage-to-stage stability:

```text
decomposition ↔ review
decomposition ↔ answer
evidence ↔ answer
review ↔ answer
```

Notebook 04 adds feedback:

```text
case
  → compute phase-lock alignments
  → identify weakest stage-pair
  → residual = gate − observed cosine
  → apply targeted revision
  → recompute alignment
  → compare before/after
```

Core update rule:

```text
v_revised = normalize((1 − alpha) v_original + alpha v_target)
```

## 0. Environment setup

In [ ]:
import os, re, math, json, random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 9423
random.seed(SEED)
np.random.seed(SEED)

REPO_NAME = "temporal-phase-lock"
NOTEBOOK_ID = "04_residual_feedback_loop"

BASE_DIR = Path(".")
FIG_DIR = BASE_DIR / "figures"
DOCS_DIR = BASE_DIR / "docs"
ARTIFACT_DIR = BASE_DIR / "artifacts"

for d in [FIG_DIR, DOCS_DIR, ARTIFACT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PHASE_LOCK_GATE = 1 / math.sqrt(1**2 + 1**2)
NEAR_BOUNDARY_MARGIN = 0.05

print(f"Ready: {REPO_NAME}/{NOTEBOOK_ID}")
print(f"Phase-lock gate: {PHASE_LOCK_GATE:.6f}")

## 1. Recreate cases and parsing utilities

In [ ]:
@dataclass
class TemporalCase:
    case_id: str
    question: str
    evidence: List[str]
    gold: str
    answer_type: str

CASES = [
    TemporalCase("TQA-001","Who joined after Mira but before Theo?",
        ["Mira joined the project in March.","Jon joined the project in May.","Theo joined the project in August.","Leah joined the project in November."],
        "Jon","between"),
    TemporalCase("TQA-002","Which event happened first?",
        ["The calibration finished on Tuesday.","The sensor test started on Monday.","The report was drafted on Wednesday."],
        "sensor test","first"),
    TemporalCase("TQA-003","What happened immediately after the review?",
        ["The draft happened before the review.","The review happened before the revision.","The revision happened before the release."],
        "revision","after"),
    TemporalCase("TQA-004","Which item was completed last?",
        ["Notebook 01 was completed before Notebook 02.","Notebook 03 was completed after Notebook 02.","Notebook 04 was completed after Notebook 03."],
        "Notebook 04","last"),
    TemporalCase("TQA-005","Did the audit occur before deployment?",
        ["The prototype was tested in April.","The audit occurred in June.","Deployment occurred in September."],
        "Yes","yes_no"),
]

MONTH_ORDER = {"january":1,"february":2,"march":3,"april":4,"may":5,"june":6,"july":7,"august":8,"september":9,"october":10,"november":11,"december":12}
WEEKDAY_ORDER = {"monday":1,"tuesday":2,"wednesday":3,"thursday":4,"friday":5,"saturday":6,"sunday":7}

def clean_phrase(text: str) -> str:
    text = re.sub(r"[\?\.]$", "", text.strip())
    text = re.sub(r"^(the|a|an)\s+", "", text, flags=re.I)
    return text

def extract_subject(sentence: str) -> str:
    parts = re.split(r"\s+(joined|occurred|finished|started|was|were|completed|happened|drafted|tested)\b",
                     sentence, maxsplit=1, flags=re.I)
    return clean_phrase(parts[0])

def detect_operators(question: str) -> List[str]:
    q = question.lower()
    ops = []
    if q.startswith("did") and "before" in q: ops.append("yes_no_before")
    if "after" in q and "before" in q: ops.append("between")
    if "immediately after" in q: ops.append("immediately_after")
    if "happened first" in q or "event happened first" in q or " first" in q: ops.append("first")
    if "completed last" in q or " last" in q: ops.append("last")
    for op in ["immediately after", "after", "before", "between", "first", "last"]:
        tag = op.replace(" ", "_")
        if op in q and tag not in ops:
            ops.append(tag)
    return list(dict.fromkeys(ops))

def extract_question_anchors(question: str) -> Dict:
    q = question.strip()
    ql = q.lower()
    m = re.search(r"after\s+([A-Za-z0-9 ]+?)\s+but\s+before\s+([A-Za-z0-9 ]+)\?", q, flags=re.I)
    if m:
        return {"anchors":[clean_phrase(m.group(1)), clean_phrase(m.group(2))],
                "target":"entity_between_anchors","relation":"after_left_and_before_right","required_anchor_count":2}
    m = re.search(r"immediately after\s+(.+?)\?", q, flags=re.I)
    if m:
        return {"anchors":[clean_phrase(m.group(1))],
                "target":"successor_event","relation":"immediate_successor","required_anchor_count":1}
    m = re.match(r"did\s+(.+?)\s+occur\s+before\s+(.+?)\?", q, flags=re.I)
    if m:
        return {"anchors":[clean_phrase(m.group(1)), clean_phrase(m.group(2))],
                "target":"yes_no_comparison","relation":"before","required_anchor_count":2}
    if "first" in ql:
        return {"anchors":["timeline_start"],"target":"earliest_event","relation":"min_time","required_anchor_count":1}
    if "last" in ql:
        return {"anchors":["timeline_end"],"target":"latest_event","relation":"max_time","required_anchor_count":1}
    return {"anchors":[],"target":"unknown","relation":"unknown","required_anchor_count":0}

def extract_time_fact(sentence: str):
    s = sentence.lower()
    for month, idx in MONTH_ORDER.items():
        if month in s:
            return extract_subject(sentence), idx, month.title()
    for day, idx in WEEKDAY_ORDER.items():
        if day in s:
            return extract_subject(sentence), idx, day.title()
    return None

def extract_directed_edges(sentence: str):
    s = sentence.strip().rstrip(".")
    patterns = [
        (r"(.+?)\s+was completed before\s+(.+)", False),
        (r"(.+?)\s+was completed after\s+(.+)", True),
        (r"(.+?)\s+happened before\s+(.+)", False),
        (r"(.+?)\s+happened after\s+(.+)", True),
    ]
    for pat, reverse in patterns:
        m = re.match(pat, s, flags=re.I)
        if m:
            a, b = clean_phrase(m.group(1)), clean_phrase(m.group(2))
            return [(b, "before", a)] if reverse else [(a, "before", b)]
    return []

def topological_order_from_edges(edges):
    nodes = sorted(set([a for a,_,b in edges] + [b for a,_,b in edges]))
    incoming = {n:set() for n in nodes}
    outgoing = {n:set() for n in nodes}
    for a, rel, b in edges:
        if rel == "before":
            outgoing[a].add(b); incoming[b].add(a)
    order, ready = [], sorted([n for n in nodes if not incoming[n]])
    while ready:
        n = ready.pop(0); order.append(n)
        for m in sorted(outgoing[n]):
            incoming[m].discard(n)
            if not incoming[m]: ready.append(m)
        ready = sorted(set(ready))
    return order

def decompose_evidence(case: TemporalCase) -> Dict:
    facts, labels, explicit_edges, rows = {}, {}, [], []
    for sent in case.evidence:
        tf = extract_time_fact(sent)
        if tf:
            subject, idx, label = tf
            facts[subject] = idx
            labels[subject] = label
            rows.append({"source": subject, "relation": "has_time", "target": label, "edge_type": "fact"})
        for a, rel, b in extract_directed_edges(sent):
            explicit_edges.append((a, rel, b))
            rows.append({"source": a, "relation": rel, "target": b, "edge_type": "constraint"})
    derived_edges = []
    if facts:
        timeline = [k for k,v in sorted(facts.items(), key=lambda kv: kv[1])]
        for a, b in zip(timeline[:-1], timeline[1:]):
            derived_edges.append((a, "before", b))
            rows.append({"source": a, "relation": "before", "target": b, "edge_type": "derived"})
    else:
        timeline = topological_order_from_edges(explicit_edges) if explicit_edges else []
    return {"facts": facts, "labels": labels, "explicit_edges": explicit_edges,
            "derived_edges": derived_edges, "all_edges": explicit_edges + derived_edges,
            "timeline": timeline, "rows": rows}

def loose_contains(name: str, collection: List[str]) -> bool:
    n = name.lower().strip()
    return any(n in item.lower() or item.lower() in n for item in collection)

## 2. Baseline phase-lock run

In [ ]:
def reformulate(question: str) -> Dict:
    ops = detect_operators(question)
    if "between" in ops: return {"intent":"between"}
    if "first" in ops: return {"intent":"first"}
    if "immediately_after" in ops: return {"intent":"after"}
    if "last" in ops: return {"intent":"last"}
    if "yes_no_before" in ops: return {"intent":"yes_no_before"}
    return {"intent":"unknown"}

def answer_from_trace(case: TemporalCase, rewritten: Dict) -> str:
    intent = reformulate(case.question)["intent"]
    timeline = rewritten["timeline"]
    facts = rewritten["facts"]
    if intent == "between":
        anchors = extract_question_anchors(case.question)["anchors"]
        if len(anchors) == 2 and anchors[0] in facts and anchors[1] in facts:
            between = [name for name, t in facts.items() if facts[anchors[0]] < t < facts[anchors[1]]]
            return between[0] if between else "UNKNOWN"
        return "UNKNOWN"
    if intent == "first":
        return timeline[0] if timeline else "UNKNOWN"
    if intent == "after":
        matches = [i for i, item in enumerate(timeline) if "review" in item.lower()]
        if matches and matches[0] + 1 < len(timeline):
            return timeline[matches[0] + 1]
        return "UNKNOWN"
    if intent == "last":
        return timeline[-1] if timeline else "UNKNOWN"
    if intent == "yes_no_before":
        anchors = extract_question_anchors(case.question)["anchors"]
        if len(anchors) == 2:
            a_key = next((k for k in facts if anchors[0].lower() in k.lower() or k.lower() in anchors[0].lower()), None)
            b_key = next((k for k in facts if anchors[1].lower() in k.lower() or k.lower() in anchors[1].lower()), None)
            if a_key and b_key:
                return "Yes" if facts[a_key] < facts[b_key] else "No"
        return "UNKNOWN"
    return "UNKNOWN"

def review_answer(pred: str, gold: str, rewritten: Dict) -> Dict:
    pred_norm = pred.lower().strip()
    gold_norm = gold.lower().strip()
    exact = pred_norm == gold_norm or pred_norm in gold_norm or gold_norm in pred_norm
    supported = pred != "UNKNOWN" and (pred in rewritten["timeline"] or pred in rewritten["facts"] or pred in ["Yes", "No"])
    return {
        "exact_match": int(bool(exact)),
        "supported": int(bool(supported)),
        "answer_known": int(pred != "UNKNOWN"),
        "review_status": "PASS" if exact and supported else "CHECK",
    }

def decomposition_features(case: TemporalCase) -> Dict:
    qa = extract_question_anchors(case.question)
    ev = decompose_evidence(case)
    required = qa["anchors"]
    required_n = qa["required_anchor_count"]
    timeline = ev["timeline"]
    edges = ev["all_edges"]
    if required in (["timeline_start"], ["timeline_end"]):
        recovered_n = 1 if timeline else 0
    else:
        recovered_n = sum(1 for a in required if loose_contains(a, timeline))
    anchor_score = recovered_n / required_n if required_n else 1.0
    min_edges = 2 if case.answer_type == "between" else 1
    constraint_coverage = min(1.0, len(edges) / min_edges) if min_edges else 1.0
    return {
        "anchor_score": anchor_score,
        "constraint_coverage": constraint_coverage,
        "recovered_edges": len(edges),
        "operator_count": len(detect_operators(case.question)),
        "timeline_length": len(timeline),
    }

def evidence_features(case: TemporalCase) -> Dict:
    ev = decompose_evidence(case)
    total_edges = len(ev["all_edges"])
    return {
        "fact_count": len(ev["facts"]),
        "explicit_edge_count": len(ev["explicit_edges"]),
        "derived_edge_count": len(ev["derived_edges"]),
        "total_edges": total_edges,
        "evidence_density": total_edges / max(1, len(case.evidence)),
    }

def review_features(case: TemporalCase, baseline_row: Dict) -> Dict:
    ambiguity_flag = 0 if baseline_row["review_status"] == "PASS" else 1
    return {
        "exact_match": baseline_row["exact_match"],
        "supported": baseline_row["supported"],
        "answer_known": baseline_row["answer_known"],
        "ambiguity_flag": ambiguity_flag,
        "evidence_density": baseline_row["edge_count"] / max(1, baseline_row["evidence_count"]),
    }

def answer_features(case: TemporalCase, baseline_row: Dict) -> Dict:
    pred = str(baseline_row["prediction"])
    gold = str(baseline_row["gold"])
    return {
        "answer_known": int(pred != "UNKNOWN"),
        "prediction_token_length": 0 if pred == "UNKNOWN" else len(pred.split()),
        "gold_token_length": len(gold.split()),
        "answer_type_id": {"between":1, "first":2, "after":3, "last":4, "yes_no":5}.get(case.answer_type, 0),
        "is_yes_no": int(case.answer_type == "yes_no"),
    }

def safe_norm(v: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(v)
    return v if n == 0 else v / n

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    aa, bb = safe_norm(a), safe_norm(b)
    if np.linalg.norm(aa) == 0 or np.linalg.norm(bb) == 0:
        return 0.0
    return float(np.dot(aa, bb))

ALIGNMENT_PAIRS = [
    ("decomposition", "review"),
    ("decomposition", "answer"),
    ("evidence", "answer"),
    ("review", "answer"),
]

def build_stage_vectors(case: TemporalCase) -> Tuple[Dict[str, np.ndarray], Dict]:
    ev = decompose_evidence(case)
    pred = answer_from_trace(case, ev)
    rev = review_answer(pred, case.gold, ev)
    baseline = {
        "case_id": case.case_id,
        "prediction": pred,
        "gold": case.gold,
        "timeline": " → ".join(ev["timeline"]),
        "timeline_length": len(ev["timeline"]),
        "edge_count": len(ev["all_edges"]),
        "evidence_count": len(case.evidence),
        **rev
    }
    vectors = {
        "decomposition": np.array(list(decomposition_features(case).values()), dtype=float),
        "evidence": np.array(list(evidence_features(case).values()), dtype=float),
        "review": np.array(list(review_features(case, baseline).values()), dtype=float),
        "answer": np.array(list(answer_features(case, baseline).values()), dtype=float),
    }
    return vectors, baseline

def compute_alignments(vectors: Dict[str, np.ndarray], case_id: str, pass_name: str) -> pd.DataFrame:
    rows = []
    for left, right in ALIGNMENT_PAIRS:
        c = cosine(vectors[left], vectors[right])
        margin = c - PHASE_LOCK_GATE
        if c >= PHASE_LOCK_GATE:
            label = "phase_locked"
        elif c >= PHASE_LOCK_GATE - NEAR_BOUNDARY_MARGIN:
            label = "near_boundary"
        else:
            label = "drift_candidate"
        rows.append({
            "case_id": case_id,
            "pass": pass_name,
            "alignment_pair": f"{left}_to_{right}",
            "left_stage": left,
            "right_stage": right,
            "cosine_alignment": c,
            "phase_lock_gate": PHASE_LOCK_GATE,
            "margin": margin,
            "label": label,
        })
    return pd.DataFrame(rows)

baseline_vectors = {}
baseline_rows = []
baseline_alignment_frames = []

for c in CASES:
    vectors, baseline = build_stage_vectors(c)
    baseline_vectors[c.case_id] = vectors
    baseline_rows.append(baseline)
    baseline_alignment_frames.append(compute_alignments(vectors, c.case_id, "baseline"))

baseline_df = pd.DataFrame(baseline_rows)
baseline_alignment_df = pd.concat(baseline_alignment_frames, ignore_index=True)

baseline_alignment_df

## 3. Weakest-pair detection and residual diagnosis

In [ ]:
DIAGNOSIS_MAP = {
    "decomposition_to_review": "review not tracking temporal structure",
    "decomposition_to_answer": "answer not tracking decomposition",
    "evidence_to_answer": "answer not grounded in evidence",
    "review_to_answer": "final answer not tracking review",
}

weakest_df = (
    baseline_alignment_df
    .sort_values(["case_id", "cosine_alignment"])
    .groupby("case_id")
    .first()
    .reset_index()
)

weakest_df["positive_residual"] = (PHASE_LOCK_GATE - weakest_df["cosine_alignment"]).clip(lower=0)
weakest_df["diagnosis"] = weakest_df["alignment_pair"].map(DIAGNOSIS_MAP)
weakest_df[["case_id", "alignment_pair", "cosine_alignment", "positive_residual", "label", "diagnosis"]]

## 4. Targeted residual revision rule

In [ ]:
def revision_alpha(residual: float) -> float:
    # Always apply at least a light revision; stronger when below gate.
    return min(0.5, max(0.1, float(residual)))

def apply_revision(vectors: Dict[str, np.ndarray], weakest_pair: str, residual: float) -> Tuple[Dict[str, np.ndarray], Dict]:
    revised = {k: v.copy() for k, v in vectors.items()}
    alpha = revision_alpha(residual)

    if weakest_pair == "evidence_to_answer":
        target_stage, source_stage = "answer", "evidence"
    elif weakest_pair == "decomposition_to_answer":
        target_stage, source_stage = "answer", "decomposition"
    elif weakest_pair == "decomposition_to_review":
        target_stage, source_stage = "review", "decomposition"
    elif weakest_pair == "review_to_answer":
        target_stage, source_stage = "answer", "review"
    else:
        target_stage, source_stage = "answer", "evidence"

    original = revised[target_stage]
    target = revised[source_stage]
    revised[target_stage] = safe_norm((1 - alpha) * safe_norm(original) + alpha * safe_norm(target))

    meta = {
        "target_stage_revised": target_stage,
        "source_stage_used": source_stage,
        "alpha": alpha,
    }
    return revised, meta

revision_meta_rows = []
revised_alignment_frames = []

for row in weakest_df.itertuples(index=False):
    vectors = baseline_vectors[row.case_id]
    revised_vectors, meta = apply_revision(vectors, row.alignment_pair, row.positive_residual)
    revised_alignment_frames.append(compute_alignments(revised_vectors, row.case_id, "revised"))
    revision_meta_rows.append({
        "case_id": row.case_id,
        "weakest_pair": row.alignment_pair,
        "diagnosis": row.diagnosis,
        "baseline_weakest_alignment": row.cosine_alignment,
        "baseline_positive_residual": row.positive_residual,
        **meta
    })

revision_meta_df = pd.DataFrame(revision_meta_rows)
revised_alignment_df = pd.concat(revised_alignment_frames, ignore_index=True)

revision_meta_df

## 5. Before/after feedback summary

In [ ]:
comparison_rows = []

for row in revision_meta_df.itertuples(index=False):
    case_id = row.case_id
    pair = row.weakest_pair

    before = baseline_alignment_df[
        (baseline_alignment_df["case_id"] == case_id) &
        (baseline_alignment_df["alignment_pair"] == pair)
    ].iloc[0]

    after = revised_alignment_df[
        (revised_alignment_df["case_id"] == case_id) &
        (revised_alignment_df["alignment_pair"] == pair)
    ].iloc[0]

    residual_before = max(0.0, PHASE_LOCK_GATE - before["cosine_alignment"])
    residual_after = max(0.0, PHASE_LOCK_GATE - after["cosine_alignment"])

    comparison_rows.append({
        "case_id": case_id,
        "weakest_pair": pair,
        "diagnosis": row.diagnosis,
        "target_stage_revised": row.target_stage_revised,
        "source_stage_used": row.source_stage_used,
        "alpha": row.alpha,
        "before_alignment": before["cosine_alignment"],
        "after_alignment": after["cosine_alignment"],
        "alignment_delta": after["cosine_alignment"] - before["cosine_alignment"],
        "residual_before": residual_before,
        "residual_after": residual_after,
        "residual_reduction": residual_before - residual_after,
        "fixed": bool(after["cosine_alignment"] >= PHASE_LOCK_GATE),
    })

feedback_summary_df = pd.DataFrame(comparison_rows)
feedback_summary_df

## 6. Figure 1 — before/after weakest-pair alignment

In [ ]:
x = np.arange(len(feedback_summary_df))
width = 0.38

plt.figure(figsize=(10, 5.2))
plt.bar(x - width/2, feedback_summary_df["before_alignment"], width, label="before")
plt.bar(x + width/2, feedback_summary_df["after_alignment"], width, label="after")
plt.axhline(PHASE_LOCK_GATE, linestyle="--", linewidth=1.8, label="45° phase-lock gate")
plt.xticks(x, feedback_summary_df["case_id"])
plt.ylim(0, 1.05)
plt.xlabel("Case")
plt.ylabel("Cosine alignment")
plt.title("Weakest-pair alignment before vs after residual feedback")
plt.legend()
plt.tight_layout()

fig1_path = FIG_DIR / "04_before_after_alignment.png"
plt.savefig(fig1_path, dpi=180, bbox_inches="tight")
plt.show()
fig1_path

## 7. Figure 2 — residual reduction

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(feedback_summary_df["case_id"], feedback_summary_df["residual_reduction"])
plt.axhline(0, linewidth=1)
plt.xlabel("Case")
plt.ylabel("Residual reduction")
plt.title("Residual reduction after targeted feedback")
plt.tight_layout()

fig2_path = FIG_DIR / "04_residual_reduction.png"
plt.savefig(fig2_path, dpi=180, bbox_inches="tight")
plt.show()
fig2_path

## 8. Figure 3 — feedback loop diagram

In [ ]:
stages = ["detect", "diagnose", "revise", "remeasure"]
x = np.arange(len(stages))
y = np.zeros(len(stages))

plt.figure(figsize=(9.5, 2.8))
plt.plot(x, y, marker="o", linewidth=2)
for i, label in enumerate(stages):
    plt.text(i, 0.08, label, ha="center", va="bottom", fontsize=12)
    if i < len(stages) - 1:
        plt.annotate("", xy=(i + 0.82, 0), xytext=(i + 0.18, 0),
                     arrowprops=dict(arrowstyle="->", lw=1.6))
plt.text(1.5, -0.13, "residual = gate − observed cosine", ha="center", va="top", fontsize=10)
plt.ylim(-0.3, 0.35)
plt.yticks([]); plt.xticks([])
plt.title("Residual feedback loop")
plt.tight_layout()

fig3_path = FIG_DIR / "04_feedback_loop_diagram.png"
plt.savefig(fig3_path, dpi=180, bbox_inches="tight")
plt.show()
fig3_path

## 9. Full before/after alignment table

In [ ]:
full_compare = baseline_alignment_df.merge(
    revised_alignment_df,
    on=["case_id", "alignment_pair", "left_stage", "right_stage", "phase_lock_gate"],
    suffixes=("_before", "_after")
)

full_compare["alignment_delta"] = (
    full_compare["cosine_alignment_after"] -
    full_compare["cosine_alignment_before"]
)

full_compare = full_compare[[
    "case_id", "alignment_pair",
    "cosine_alignment_before", "cosine_alignment_after",
    "alignment_delta", "label_before", "label_after"
]]

full_compare

## 10. Save artifacts

In [ ]:
baseline_path = ARTIFACT_DIR / "04_residual_feedback_baseline.csv"
revised_path = ARTIFACT_DIR / "04_residual_feedback_revised.csv"
summary_path = ARTIFACT_DIR / "04_residual_feedback_summary.csv"
full_compare_path = ARTIFACT_DIR / "04_residual_feedback_full_compare.csv"
doc_summary_path = DOCS_DIR / "04_residual_feedback_loop_summary.md"

baseline_alignment_df.to_csv(baseline_path, index=False)
revised_alignment_df.to_csv(revised_path, index=False)
feedback_summary_df.to_csv(summary_path, index=False)
full_compare.to_csv(full_compare_path, index=False)

fixed_count = int(feedback_summary_df["fixed"].sum())
mean_before = feedback_summary_df["before_alignment"].mean()
mean_after = feedback_summary_df["after_alignment"].mean()
mean_delta = feedback_summary_df["alignment_delta"].mean()
mean_residual_reduction = feedback_summary_df["residual_reduction"].mean()

summary_md = f"""# Notebook 04 — Residual Feedback Loop

Repo: `{REPO_NAME}`  
Notebook: `notebooks/04_residual_feedback_loop.ipynb`

## Purpose

Detect weakest temporal-reasoning stage, revise it, and re-measure phase-lock.

Core loop:

```text
detect → diagnose → revise → remeasure
```

Canonical gate:

```text
cos(theta) >= 1 / sqrt(1^2 + 1^2)
```

Numerical gate:

```text
{PHASE_LOCK_GATE:.6f}
```

## Summary metrics

| Metric | Value |
|---|---:|
| Mean weakest-pair alignment before | {mean_before:.3f} |
| Mean weakest-pair alignment after | {mean_after:.3f} |
| Mean alignment delta | {mean_delta:.3f} |
| Mean residual reduction | {mean_residual_reduction:.3f} |
| Fixed weakest pairs | {fixed_count} |
| Number of cases | {len(feedback_summary_df)} |

## Figures

- `figures/04_before_after_alignment.png`
- `figures/04_residual_reduction.png`
- `figures/04_feedback_loop_diagram.png`

## Artifacts

- `artifacts/04_residual_feedback_baseline.csv`
- `artifacts/04_residual_feedback_revised.csv`
- `artifacts/04_residual_feedback_summary.csv`
- `artifacts/04_residual_feedback_full_compare.csv`

## Handoff to Notebook 05

Notebook 05 should scale from transparent examples to a generated benchmark grid:

```text
operator type × ambiguity level × evidence density × drift injection
```
"""

doc_summary_path.write_text(summary_md)

print(baseline_path)
print(revised_path)
print(summary_path)
print(full_compare_path)
print(doc_summary_path)

## 11. Handoff to Notebook 05

In [ ]:
next_notebook = {
    "notebook": "05_adaptive_temporal_reasoning_benchmark.ipynb",
    "purpose": "scale from 5 transparent examples to a generated benchmark grid",
    "grid_axes": [
        "operator_type",
        "ambiguity_level",
        "evidence_density",
        "drift_injection"
    ],
    "reuse_from_04": [
        "phase_lock_gate",
        "weakest_pair_detection",
        "residual_feedback_update",
        "before_after_metrics"
    ]
}
print(json.dumps(next_notebook, indent=2))

## 12. Final notebook status

In [ ]:
status = {
    "repo": REPO_NAME,
    "notebook": NOTEBOOK_ID,
    "phase_lock_gate": PHASE_LOCK_GATE,
    "cases": len(CASES),
    "mean_before_alignment": round(float(mean_before), 3),
    "mean_after_alignment": round(float(mean_after), 3),
    "mean_alignment_delta": round(float(mean_delta), 3),
    "mean_residual_reduction": round(float(mean_residual_reduction), 3),
    "fixed_weakest_pairs": fixed_count,
    "artifacts": [
        str(baseline_path),
        str(revised_path),
        str(summary_path),
        str(full_compare_path),
        str(doc_summary_path),
        str(fig1_path),
        str(fig2_path),
        str(fig3_path),
    ],
}
print(json.dumps(status, indent=2))